# Novel-view evaluation: metrics vs. papers + genuinely novel angles

Two things this notebook does, both about comparing your `GS3D` / `DSYG`
(and `GS3D_Ratio` / `GS3D_Sampled` if you've added them) reconstructions
against the real scene:

1. **Quantitative**: render every held-out test camera (never seen during
   3DGS training) and compare against the real photograph with PSNR/SSIM/
   LPIPS, using the same train/test split convention the papers you're
   citing use (every 8th image, sorted by filename -- the convention from
   Mip-NeRF360, reused by 3DGS, GOF, and most follow-ups including,
   presumably, Don't Splat Your Gaussians). Matching the convention is
   what makes your numbers *comparable* to the tables in those papers, not
   just internally consistent between your own methods.
2. **Qualitative**: a free-orbit slider viewer for angles that have **no
   photograph at all** -- this is where you actually see floaters,
   blurring, or geometry that only looks right near a training camera.
   None of that shows up in the held-out table above, because every
   held-out view is still relatively close to *some* training view.

This notebook is self-contained and Colab-first (headless Xvfb + virtual
display, matching your other notebook's setup). A **separate script**,
`interactive_realtime_viewer.py`, exists alongside `rdv` for later --
when you have a local machine with a real GPU and a real display and want
a genuinely interactive (mouse+keyboard, every frame) flythrough. That one
can't run here: Colab has no attached display to stream to.

**Before you run anything else**, edit the `SCENE_NAME` / `SCENE_ROOT`
cell below to point at your scene's folder, and make sure it has an
`images/` subfolder with the ORIGINAL training photographs -- standard
3DGS output ships these next to `point_cloud.ply` and `cameras.json`.
Without the photos, Part 1 (the actual metrics) can't run at all.


## 0. Environment setup

In [ ]:
import os
import sys

try:
    # 1. Check if we are in Colab
    import google.colab
    print("Colab environment detected. Setting up dependencies...")

    # 2. THE GPU SAFETY CHECK
    import subprocess
    if subprocess.run(['which', 'nvidia-smi']).returncode != 0:
        raise RuntimeError("NO GPU DETECTED! Please go to the top menu -> Runtime -> Change runtime type -> select T4 GPU.")

    # 3. Install system dependencies
    !sudo apt-get update -y
    !sudo apt-get install -y xvfb vulkan-tools glslang-tools vulkan-validationlayers-dev

    # Install the video library (dropping strict versioning for Python 3.12 compatibility)
    !pip install av

    # Dynamically install the exact NVIDIA GL driver
    !sudo apt-get install -y libnvidia-gl-$(nvidia-smi --query-gpu=driver_version --format=csv,noheader | cut -d. -f1)

    print("Cloning repositories...")
    if not os.path.exists('/content/vulky'):
        !git clone https://github.com/rendervous/vulky_project.git /content/vulky
    if not os.path.exists('/content/rdv'):
        !git clone https://github.com/Cdelim/rdv.git /content/rdv

    # 4. Set up the headless Vulkan environment
    os.environ['DISPLAY'] = ':99'
    os.environ['XDG_RUNTIME_DIR'] = '/tmp/runtime-dir'
    os.makedirs('/tmp/runtime-dir', exist_ok=True)
    os.environ['VK_ICD_FILENAMES'] = '/usr/share/vulkan/icd.d/nvidia_icd.json'

    # Start the invisible monitor
    subprocess.Popen(['Xvfb', ':99', '-screen', '0', '1024x768x24'])

    # 5. Tell Python where to find the source code
    sys.path.append(os.path.abspath('/content/vulky/src'))
    sys.path.append(os.path.abspath('/content/rdv/src'))

except RuntimeError as re:
    print(f"\nCRITICAL ERROR: {re}\n")
    sys.exit(1)
except ImportError:
    print("Executing locally...")
except Exception as e:
    print("Setup failed:", e)


In [ ]:
import os
# os.environ['RDV_DEBUG'] = 'True'  ## uncomment to enable debug logging from vulkan
import rdv
import torch
!pip install plyfile scikit-image lpips pandas ipywidgets -q


## 1. Point this at your scene

`images/` (the original training photographs) is the one folder the earlier
notebook never needed but this one does -- it's what the metrics are
computed against. If your 3DGS export doesn't have it, PSNR/SSIM/LPIPS
against real photos is not possible; you'd only have the synthetic-scene
checks from `evaluation_experiments.py` available to you.


In [ ]:
import numpy as np
import json
from plyfile import PlyData

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Not running in Colab -- skipping drive mount. "
          "Edit SCENE_ROOT below to point at your local copy of the scene.")

# ---- Edit these three lines for your scene ----
SCENE_NAME = "bicycle"
SCENE_ROOT = f"/content/drive/MyDrive/Gaussians/{SCENE_NAME}"

ply_path     = f"{SCENE_ROOT}/point_cloud.ply"
cameras_path = f"{SCENE_ROOT}/cameras.json"
images_dir   = f"{SCENE_ROOT}/images"

# Where thesis-ready figures/CSVs from this notebook get saved.
output_dir = f"{SCENE_ROOT}/nvs_eval_outputs"
os.makedirs(output_dir, exist_ok=True)

print(f"Loading PLY file from {ply_path} ... (this can take a few seconds for millions of points)")
print(f"Photos expected in: {images_dir}")
print(f"Outputs will be saved to: {output_dir}")


## 2. Covariance helpers (same as your other notebook)

In [ ]:
def compute_inverse_covariance(scales, rotations):
    """
    Computes the upper triangular 6 elements of the Inverse Covariance Matrix
    formula: Sigma^-1 = R * S^-2 * R^T
    """
    inv_sq_scales = 1.0 / (scales ** 2 + 1e-7)

    r = rotations[:, 0]
    x = rotations[:, 1]
    y = rotations[:, 2]
    z = rotations[:, 3]

    R = torch.zeros((rotations.shape[0], 3, 3), device=scales.device)
    R[:, 0, 0] = 1.0 - 2.0 * (y**2 + z**2)
    R[:, 0, 1] = 2.0 * (x * y - r * z)
    R[:, 0, 2] = 2.0 * (x * z + r * y)
    R[:, 1, 0] = 2.0 * (x * y + r * z)
    R[:, 1, 1] = 1.0 - 2.0 * (x**2 + z**2)
    R[:, 1, 2] = 2.0 * (y * z - r * x)
    R[:, 2, 0] = 2.0 * (x * z - r * y)
    R[:, 2, 1] = 2.0 * (y * z + r * x)
    R[:, 2, 2] = 1.0 - 2.0 * (x**2 + y**2)

    inv_cov = torch.zeros((scales.shape[0], 6), device=scales.device)
    for i in range(3):
        for j in range(i, 3):
            val = (R[:, i, 0] * inv_sq_scales[:, 0] * R[:, j, 0] +
                   R[:, i, 1] * inv_sq_scales[:, 1] * R[:, j, 1] +
                   R[:, i, 2] * inv_sq_scales[:, 2] * R[:, j, 2])
            idx = i * 3 - (i * (i + 1)) // 2 + j
            inv_cov[:, idx] = val
    return inv_cov


def compute_covariance(scales, rotations):
    """
    Computes the upper triangular 6 elements of the Covariance Matrix
    formula: Sigma = R * S^2 * R^T
    """
    sq_scales = scales ** 2

    r = rotations[:, 0]
    x = rotations[:, 1]
    y = rotations[:, 2]
    z = rotations[:, 3]

    R = torch.zeros((rotations.shape[0], 3, 3), device=scales.device)
    R[:, 0, 0] = 1.0 - 2.0 * (y**2 + z**2)
    R[:, 0, 1] = 2.0 * (x * y - r * z)
    R[:, 0, 2] = 2.0 * (x * z + r * y)
    R[:, 1, 0] = 2.0 * (x * y + r * z)
    R[:, 1, 1] = 1.0 - 2.0 * (x**2 + z**2)
    R[:, 1, 2] = 2.0 * (y * z - r * x)
    R[:, 2, 0] = 2.0 * (x * z - r * y)
    R[:, 2, 1] = 2.0 * (y * z + r * x)
    R[:, 2, 2] = 1.0 - 2.0 * (x**2 + y**2)

    cov = torch.zeros((scales.shape[0], 6), device=scales.device)
    for i in range(3):
        for j in range(i, 3):
            val = (R[:, i, 0] * sq_scales[:, 0] * R[:, j, 0] +
                   R[:, i, 1] * sq_scales[:, 1] * R[:, j, 1] +
                   R[:, i, 2] * sq_scales[:, 2] * R[:, j, 2])
            idx = i * 3 - (i * (i + 1)) // 2 + j
            cov[:, idx] = val
    return cov


## 3. Load the scene and build every available method's map

Builds `gs_map` (GS3D) and `dsyg_map` (DSYG) always, and `ratio_map`
(GS3D_Ratio) / `sampled_map` (GS3D_Sampled) if you've added those files to
your `rdv` project -- same `hasattr` pattern as `interactive_overlap_viewer.py`,
so this notebook doesn't break if you haven't.

`opacities_vk` is shared unmodified across all four -- checked against the
shaders directly: `decomposition_tracking_GS.h`, `DontSplashYourGaussians.h`,
`decomposition_tracking_GS_ratio.h`, and `decomposition_tracking_GS_sampled.h`
all read `opacities.data[i]` as a plain per-Gaussian alpha in [0,1] (clamped
to 0.999), so there's no per-method conversion needed here.


In [ ]:
plydata = PlyData.read(ply_path)
v = plydata['vertex']

x = torch.tensor(v['x'].copy(), dtype=torch.float32)
y = torch.tensor(v['y'].copy(), dtype=torch.float32)
z = torch.tensor(v['z'].copy(), dtype=torch.float32)
positions = torch.stack((x, y, z), dim=-1)

f_dc_0 = torch.tensor(v['f_dc_0'].copy(), dtype=torch.float32)
f_dc_1 = torch.tensor(v['f_dc_1'].copy(), dtype=torch.float32)
f_dc_2 = torch.tensor(v['f_dc_2'].copy(), dtype=torch.float32)
sh_dc = torch.stack((f_dc_0, f_dc_1, f_dc_2), dim=-1)

rest_names = [f'f_rest_{i}' for i in range(45)]
f_rest = torch.stack([torch.tensor(v[name].copy(), dtype=torch.float32) for name in rest_names], dim=-1)

scale_names = ['scale_0', 'scale_1', 'scale_2']
scales = torch.stack([torch.tensor(v[name].copy(), dtype=torch.float32) for name in scale_names], dim=-1)
scales = torch.exp(scales).clamp(min=1e-6, max=5.0)

rot_names = ['rot_0', 'rot_1', 'rot_2', 'rot_3']
rotations = torch.stack([torch.tensor(v[name].copy(), dtype=torch.float32) for name in rot_names], dim=-1)
rotations = torch.nn.functional.normalize(rotations, dim=-1)

opacities = torch.sigmoid(torch.tensor(v['opacity'].copy(), dtype=torch.float32))

# Cull near-invisible Gaussians (they add nothing visually but choke the BVH build)
mask = opacities.squeeze() > 0.02

inv_covs = compute_inverse_covariance(scales, rotations)
covs = compute_covariance(scales, rotations)

positions = positions[mask]
sh_dc = sh_dc[mask]
f_rest = f_rest[mask]
scales = scales[mask]
opacities = opacities[mask]
inv_covs = inv_covs[mask]
covs = covs[mask]

print(f"Loaded {positions.shape[0]} Gaussians after culling ({mask.numel() - mask.sum().item()} removed).")

positions_vk = rdv.tensor_copy(rdv.vec3(positions).to(rdv.device()))
colors_vk = rdv.tensor_copy(rdv.vec3(sh_dc).to(rdv.device()))
scales_vk = rdv.tensor_copy(rdv.vec3(scales).to(rdv.device()))
f_rest_vk = rdv.tensor_copy(f_rest.to(rdv.device()))
inv_covs_vk = rdv.tensor_copy(inv_covs.to(rdv.device()))
opacities_vk = rdv.tensor_copy(opacities.to(rdv.device()))
covs_vk = rdv.tensor_copy(covs.to(rdv.device()))


def _build(map_cls):
    m = map_cls(
        positions_vk, colors_vk,
        inv_covs=inv_covs_vk, opacities=opacities_vk,
        scales=scales_vk, f_rest=f_rest_vk, covs=covs_vk,
    )
    m.build_ads()
    return m


methods = {'GS3D': _build(rdv.GS3D), 'DSYG': _build(rdv.DSYG)}
if hasattr(rdv, 'GS3D_Ratio'):
    methods['GS3D_Ratio'] = _build(rdv.GS3D_Ratio)
if hasattr(rdv, 'GS3D_Sampled'):
    methods['GS3D_Sampled'] = _build(rdv.GS3D_Sampled)

print("Built methods:", list(methods.keys()))


## 4. STOP FIRST -- check what's actually in your cameras.json

The FOV/focal-length question matters a lot here: if `rdv.CameraProbes` uses
a fixed internal default FOV rather than reading one per camera, every render
below will have the WRONG field of view relative to the real photos, and
PSNR/SSIM/LPIPS will be quietly wrong without throwing any error. Run this
and actually look at the printed keys before trusting anything in Part 1.


In [ ]:
with open(cameras_path, 'r') as f:
    cameras_data = json.load(f)

print("Keys in a camera entry:", list(cameras_data[0].keys()))
print("Example entry:", cameras_data[0])
print()
fx_like = [k for k in cameras_data[0].keys() if 'f' in k.lower() and k.lower() not in ('id',)]
print("candidate focal-length-ish keys:", fx_like)
for k in fx_like:
    vals = [c.get(k) for c in cameras_data[:20]]
    print(f"  {k} across first 20 cameras: {vals}")
print()
print("If you see fx/fy/FovX/FovY here and they DIFFER across cameras, check")
print("whether rdv.CameraProbes/rdv.Sensor take a per-camera FOV (try")
print("help(rdv.CameraProbes)) before trusting the metrics below.")


## 5. Camera pose extraction + train/test split

Pose convention (forward = 3rd column of rotation, up = -2nd column) matches
your other notebook's cell 11 -- not re-derived, to avoid introducing a sign
error your existing, working code doesn't have.

Split convention: sort by filename, hold out every 8th image as test. This
is the convention used since Mip-NeRF360 and reused by 3DGS and most
follow-up papers -- matching it is what makes these numbers comparable to
the papers' tables. If a paper you're citing published an exact list of
held-out filenames (rather than just "every 8th"), prefer that list instead
for an exact match; `test_every=8` is the standard fallback when it didn't.


In [ ]:
def camera_pose_list(cam):
    pos = np.array(cam['position'], dtype=np.float32)
    rot = np.array(cam['rotation'], dtype=np.float32)
    forward = rot[:, 2]
    up = -rot[:, 1]
    target = pos + forward
    return pos, target, up


def _img_key(cam):
    for k in ('img_name', 'image_name', 'file_path', 'filename', 'image_path'):
        if k in cam:
            return cam[k]
    return str(cam.get('id', id(cam)))  # fallback: at least stable ordering


def train_test_split(cameras, test_every=8):
    ordered = sorted(cameras, key=_img_key)
    test = [c for i, c in enumerate(ordered) if i % test_every == 0]
    train = [c for i, c in enumerate(ordered) if i % test_every != 0]
    return train, test


def find_image_path(images_dir, cam, extensions=('.jpg', '.jpeg', '.png', '.JPG', '.PNG')):
    base = _img_key(cam)
    base_noext = os.path.splitext(str(base))[0]
    for ext in extensions:
        candidate = os.path.join(images_dir, base_noext + ext)
        if os.path.exists(candidate):
            return candidate
    return None


train_cams, test_cams = train_test_split(cameras_data, test_every=8)
print(f"{len(cameras_data)} total cameras -> {len(train_cams)} train / {len(test_cams)} test")
n_found = sum(1 for c in test_cams if find_image_path(images_dir, c) is not None)
print(f"Found matching photos for {n_found}/{len(test_cams)} test cameras in {images_dir}")
if n_found < len(test_cams):
    print("Some test cameras have no matching photo -- they'll be skipped with a warning in Part 1.")


In [ ]:
import math

_fov_warned = {'shown': False}


def make_camera_probes(camera_poses, width, height, cam=None):
    """
    aspect_ratio matters: perspective_camera_sensors.h scales the
    HORIZONTAL ray deflection by aspect_ratio but leaves the vertical one
    unscaled, so leaving CameraProbes at its default aspect_ratio=1.0
    stretches/squeezes any non-square render.
    """
    kwargs = dict(camera_poses=camera_poses, aspect_ratio=width / height)
    fy = cam.get('fy') if cam is not None else None
    if fy:
        # fy = vertical focal length in pixels -> exact vertical FOV, matching
        # what CameraProbes' `fov` expects (sz=1/tan(fov/2) applies uniformly
        # and sy is unscaled in perspective_camera_sensors.h's FORWARD).
        kwargs['fov'] = 2.0 * math.atan(height / (2.0 * fy))
    elif cam is not None and not _fov_warned['shown']:
        print("NOTE: this camera has no 'fy' field -- using CameraProbes' default "
              "45-degree vertical FOV. Check the 'STOP FIRST' cell above for what "
              "focal-length-ish keys your cameras.json actually has.")
        _fov_warned['shown'] = True
    return rdv.CameraProbes(**kwargs)


def make_sensor(width, height, camera_poses, cam=None):
    """
    IMPORTANT: rdv.Sensor's shape arguments (after the leading camera-count)
    are in (rows, cols) = (height, width) order, matching numpy/PIL/
    matplotlib's image convention -- NOT (width, height). Passing them as
    (width, height) silently transposes the render: a landscape photo comes
    out with width and height swapped (e.g. 1080x1920 instead of 1920x1080).

    Traced this from perspective_camera_sensors.h + capture_forward.h:
    `_input[0]` (the shader's horizontal/sx ray deflection) is generated
    from the FASTEST-varying / LAST axis of the output tensor, and that
    axis's size is controlled by Sensor's THIRD positional argument, not
    its second.
    """
    return rdv.Sensor(
        1, height, width,
        samples_location=(rdv.SampleLocation.CORNER, rdv.SampleLocation.RANDOM, rdv.SampleLocation.RANDOM),
        probes_map=make_camera_probes(camera_poses, width, height, cam),
    )


## Part 1 -- Quantitative: PSNR / SSIM / LPIPS on held-out views

PSNR is simple enough to write by hand. SSIM and LPIPS are NOT -- both have
fiddly-enough reference implementations (windowing, calibration weights)
that hand-rolling them risks numbers that don't actually match what the
papers report, which defeats the point. Use the real libraries
(`scikit-image` for SSIM, `lpips` for LPIPS -- both already `pip install`ed
above).


In [ ]:
def evaluate_on_held_out(methods, cameras_data, images_dir, test_every=8,
                          samples=64, device=None, max_test_views=None, verbose=True):
    """
    methods: {'GS3D': gs_map, 'DSYG': dsyg_map, ...} -- your already-built maps.
    Renders EVERY held-out test camera at ITS OWN resolution (not an
    arbitrary fixed size) with every method, loads the matching photo,
    computes all three metrics, and averages over the test set -- the same
    shape of table the papers report.
    """
    device = device or rdv.device()
    _, test_cams = train_test_split(cameras_data, test_every=test_every)
    if max_test_views is not None:
        test_cams = test_cams[:max_test_views]

    rows = []
    for cam in test_cams:
        img_path = find_image_path(images_dir, cam)
        if img_path is None:
            if verbose:
                print(f"WARNING: no photo found for camera {_img_key(cam)} in {images_dir} -- skipping")
            continue

        ref = np.asarray(Image.open(img_path).convert('RGB'), dtype=np.float32) / 255.0

        pos, target, up = camera_pose_list(cam)
        pose_list = list(pos) + list(target) + list(up)
        camera_poses = rdv.tensor_copy(torch.tensor(pose_list, dtype=torch.float32).reshape(1, 9))
        w, h = int(cam['width']), int(cam['height'])
        sensor = make_sensor(w, h, camera_poses, cam=cam)

        if ref.shape[0] != h or ref.shape[1] != w:
            if verbose:
                print(f"WARNING: photo is {ref.shape[1]}x{ref.shape[0]} but camera says "
                      f"{w}x{h} -- resolution mismatch, results for this view may be misleading")

        for name, model in methods.items():
            rendered = sensor.view(model, samples=samples).capture()[0]
            arr = np.clip(rendered.detach().cpu().numpy(), 0.0, 1.0)
            arr = np.flipud(arr)  # matches plt.gca().invert_yaxis() used elsewhere in your notebooks

            rows.append(dict(
                camera=_img_key(cam), method=name,
                psnr=psnr(arr, ref), ssim=ssim(arr, ref),
                lpips=lpips_score(arr, ref, device=device),
            ))

        if verbose:
            print(f"done: {_img_key(cam)}  ({len(test_cams)} test views total)")

    import pandas as pd
    df = pd.DataFrame(rows)
    summary = df.groupby('method')[['psnr', 'ssim', 'lpips']].mean()
    summary['n_views'] = df.groupby('method').size()
    return df, summary


In [ ]:
def evaluate_on_held_out(methods, cameras_data, images_dir, test_every=8,
                          samples=64, device=None, max_test_views=None, verbose=True):
    """
    methods: {'GS3D': gs_map, 'DSYG': dsyg_map, ...} -- your already-built maps.
    Renders EVERY held-out test camera at ITS OWN resolution (not an
    arbitrary fixed size) with every method, loads the matching photo,
    computes all three metrics, and averages over the test set -- the same
    shape of table the papers report.
    """
    device = device or rdv.device()
    _, test_cams = train_test_split(cameras_data, test_every=test_every)
    if max_test_views is not None:
        test_cams = test_cams[:max_test_views]

    rows = []
    for cam in test_cams:
        img_path = find_image_path(images_dir, cam)
        if img_path is None:
            if verbose:
                print(f"WARNING: no photo found for camera {_img_key(cam)} in {images_dir} -- skipping")
            continue

        ref = np.asarray(Image.open(img_path).convert('RGB'), dtype=np.float32) / 255.0

        pos, target, up = camera_pose_list(cam)
        pose_list = list(pos) + list(target) + list(up)
        camera_poses = rdv.tensor_copy(torch.tensor(pose_list, dtype=torch.float32).reshape(1, 9))
        w, h = int(cam['width']), int(cam['height'])
        sensor = rdv.Sensor(1, w, h,
            samples_location=(rdv.SampleLocation.CORNER, rdv.SampleLocation.RANDOM, rdv.SampleLocation.RANDOM),
            probes_map=rdv.CameraProbes(camera_poses=camera_poses))

        if ref.shape[0] != h or ref.shape[1] != w:
            if verbose:
                print(f"WARNING: photo is {ref.shape[1]}x{ref.shape[0]} but camera says "
                      f"{w}x{h} -- resolution mismatch, results for this view may be misleading")

        for name, model in methods.items():
            rendered = sensor.view(model, samples=samples).capture()[0]
            arr = np.clip(rendered.detach().cpu().numpy(), 0.0, 1.0)
            arr = np.flipud(arr)  # matches plt.gca().invert_yaxis() used elsewhere in your notebooks

            rows.append(dict(
                camera=_img_key(cam), method=name,
                psnr=psnr(arr, ref), ssim=ssim(arr, ref),
                lpips=lpips_score(arr, ref, device=device),
            ))

        if verbose:
            print(f"done: {_img_key(cam)}  ({len(test_cams)} test views total)")

    import pandas as pd
    df = pd.DataFrame(rows)
    summary = df.groupby('method')[['psnr', 'ssim', 'lpips']].mean()
    summary['n_views'] = df.groupby('method').size()
    return df, summary


In [ ]:
# samples=64 is a reasonable start for GS3D/GS3D_Sampled (stochastic --
# noisier at low spp); DSYG/GS3D_Ratio are deterministic and would give the
# identical answer at samples=1, so paying 64x for them here is just to keep
# a single number for the whole table. Drop max_test_views to e.g. 5 first
# to sanity-check timing before running the full test set.
per_view_df, summary_table = evaluate_on_held_out(
    methods=methods,
    cameras_data=cameras_data,
    images_dir=images_dir,
    test_every=8,
    samples=64,
    max_test_views=None,
)

print(summary_table)
# higher psnr/ssim is better, lower lpips is better -- the papers' convention.

per_view_df.to_csv(f"{output_dir}/{SCENE_NAME}_per_view_metrics.csv", index=False)
summary_table.to_csv(f"{output_dir}/{SCENE_NAME}_summary_metrics.csv")
print(f"Saved to {output_dir}")


## Part 2 -- Compare against the published numbers

Fill in `PAPER_METRICS` below from the actual tables in the two papers, for
the SAME scene you evaluated above (`bicycle` by default). Nothing is
pre-filled here on purpose -- copying a remembered number instead of reading
it off the PDF is exactly the kind of error that's invisible until someone
checks your thesis against the source. Leave any entry you don't have as
`None`; it's skipped in the table and chart below rather than plotted as 0.

Where to look:
- **3D Gaussian Splatting for Real-Time Radiance Field Rendering** (Kerbl et
  al., 2023) -- per-scene Mip-NeRF360 numbers are usually in the
  supplementary material, not the main paper's summary table. `bicycle` is
  one of the Mip-NeRF360 outdoor scenes.
- **Don't Splat your Gaussians** -- check whichever table reports
  per-scene (not just averaged) PSNR/SSIM/LPIPS, ideally on the same
  Mip-NeRF360 scenes so the comparison is apples-to-apples.

If a paper only reports an averaged number across all Mip-NeRF360 scenes
(not `bicycle` specifically), note that in a comment -- it's still useful
context, just not a same-scene comparison.


In [ ]:
# ---- Fill these in from the papers. Leave as None if you don't have the number. ----
PAPER_METRICS = {
    "3DGS (Kerbl et al. 2023)": {
        "psnr": None,   # TODO: fill in from the paper's bicycle-scene (or Mip-NeRF360) table
        "ssim": None,   # TODO
        "lpips": None,  # TODO
    },
    "Don't Splat Your Gaussians": {
        "psnr": None,   # TODO
        "ssim": None,   # TODO
        "lpips": None,  # TODO
    },
}


def build_comparison_table(summary_table, paper_metrics):
    import pandas as pd
    rows = []
    for method, row in summary_table[['psnr', 'ssim', 'lpips']].iterrows():
        rows.append({'source': f'ours: {method}', 'psnr': row['psnr'], 'ssim': row['ssim'], 'lpips': row['lpips']})
    for paper, m in paper_metrics.items():
        rows.append({'source': paper, 'psnr': m.get('psnr'), 'ssim': m.get('ssim'), 'lpips': m.get('lpips')})
    return pd.DataFrame(rows).set_index('source')


comparison_table = build_comparison_table(summary_table, PAPER_METRICS)
print(comparison_table)

missing = [paper for paper, m in PAPER_METRICS.items() if all(v is None for v in m.values())]
if missing:
    print()
    print(f"NOTE: no numbers filled in yet for: {', '.join(missing)} -- "
          f"those rows will show as NaN / be skipped in the chart below.")

comparison_table.to_csv(f"{output_dir}/{SCENE_NAME}_comparison_table.csv")


In [ ]:
import matplotlib.pyplot as plt

def plot_comparison_table(comparison_table):
    metrics = ['psnr', 'ssim', 'lpips']
    titles = {'psnr': 'PSNR (higher better)', 'ssim': 'SSIM (higher better)', 'lpips': 'LPIPS (lower better)'}
    colors = plt.get_cmap('tab10').colors

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    for ax, metric in zip(axes, metrics):
        col = comparison_table[metric].dropna()
        if col.empty:
            ax.set_title(f"{titles[metric]}\n(no data)")
            ax.axis('off')
            continue
        bar_colors = [colors[i % len(colors)] for i in range(len(col))]
        ax.bar(range(len(col)), col.values, color=bar_colors)
        ax.set_xticks(range(len(col)))
        ax.set_xticklabels(col.index, rotation=35, ha='right', fontsize=8)
        ax.set_title(titles[metric])
        ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{output_dir}/{SCENE_NAME}_comparison_chart.png", dpi=150)
    plt.show()

plot_comparison_table(comparison_table)


import io
try:
    import ipywidgets as widgets
except ImportError:
    import subprocess as _subprocess
    _subprocess.run([sys.executable, '-m', 'pip', 'install', 'ipywidgets', '-q'])
    import ipywidgets as widgets
from IPython.display import display


def scene_orbit_defaults(positions_cpu):
    """Robust-ish centroid/radius from the raw point cloud, ignoring the
    long tail of COLMAP outlier points a straight min/max would be thrown off by."""
    centroid = positions_cpu.median(dim=0).values
    dists = (positions_cpu - centroid).norm(dim=-1)
    radius = torch.quantile(dists, 0.85).item()
    return centroid.numpy(), max(radius, 0.1)


def orbit_pose(centroid, az_deg, el_deg, dist):
    az, el = np.radians(az_deg), np.radians(el_deg)
    offset = dist * np.array([np.cos(el) * np.sin(az), np.sin(el), np.cos(el) * np.cos(az)])
    return centroid + offset, centroid, np.array([0.0, 1.0, 0.0], dtype=np.float32)


def render_novel_view(model, centroid, az, el, dist, samples, width, height):
    pos, tgt, up = orbit_pose(centroid, az, el, dist)
    pose_list = list(pos) + list(tgt) + list(up)
    camera_poses = rdv.tensor_copy(torch.tensor(pose_list, dtype=torch.float32).reshape(1, 9))
    sensor = make_sensor(width, height, camera_poses)
    img = sensor.view(model, samples=int(samples)).capture()[0]
    return np.flipud(np.clip(img.detach().cpu().numpy(), 0.0, 1.0))


In [ ]:
import io
try:
    import ipywidgets as widgets
except ImportError:
    import subprocess as _subprocess
    _subprocess.run([sys.executable, '-m', 'pip', 'install', 'ipywidgets', '-q'])
    import ipywidgets as widgets
from IPython.display import display


def scene_orbit_defaults(positions_cpu):
    """Robust-ish centroid/radius from the raw point cloud, ignoring the
    long tail of COLMAP outlier points a straight min/max would be thrown off by."""
    centroid = positions_cpu.median(dim=0).values
    dists = (positions_cpu - centroid).norm(dim=-1)
    radius = torch.quantile(dists, 0.85).item()
    return centroid.numpy(), max(radius, 0.1)


def orbit_pose(centroid, az_deg, el_deg, dist):
    az, el = np.radians(az_deg), np.radians(el_deg)
    offset = dist * np.array([np.cos(el) * np.sin(az), np.sin(el), np.cos(el) * np.cos(az)])
    return centroid + offset, centroid, np.array([0.0, 1.0, 0.0], dtype=np.float32)


def render_novel_view(model, centroid, az, el, dist, samples, width, height):
    pos, tgt, up = orbit_pose(centroid, az, el, dist)
    pose_list = list(pos) + list(tgt) + list(up)
    camera_poses = rdv.tensor_copy(torch.tensor(pose_list, dtype=torch.float32).reshape(1, 9))
    sensor = rdv.Sensor(1, width, height,
        samples_location=(rdv.SampleLocation.CORNER, rdv.SampleLocation.RANDOM, rdv.SampleLocation.RANDOM),
        probes_map=rdv.CameraProbes(camera_poses=camera_poses))
    img = sensor.view(model, samples=int(samples)).capture()[0]
    return np.flipud(np.clip(img.detach().cpu().numpy(), 0.0, 1.0))


In [ ]:
def launch_scene_viewer(methods, positions_cpu, width=420, height=420):
    centroid, radius = scene_orbit_defaults(positions_cpu)

    def to_png_bytes(arr):
        pil_img = Image.fromarray((arr * 255).astype(np.uint8), mode='RGB')
        buf = io.BytesIO()
        pil_img.save(buf, format='PNG')
        return buf.getvalue()

    az_s = widgets.FloatSlider(value=0, min=0, max=360, step=2, description='azimuth', continuous_update=False)
    el_s = widgets.FloatSlider(value=10, min=-80, max=80, step=2, description='elevation', continuous_update=False)
    dist_s = widgets.FloatSlider(value=radius * 2.0, min=radius * 0.3, max=radius * 6.0, step=radius * 0.05,
                                  description='distance', continuous_update=False)
    samples_s = widgets.IntSlider(value=32, min=1, max=512, description='samples', continuous_update=False)
    method_dd = widgets.Dropdown(options=list(methods.keys()), description='method')
    save_btn = widgets.Button(description='save snapshot (all methods)')
    img_w = widgets.Image(format='png', width=width, height=height)
    status = widgets.Label(value='')

    def _update(*_):
        status.value = 'rendering...'
        try:
            arr = render_novel_view(methods[method_dd.value], centroid,
                                     az_s.value, el_s.value, dist_s.value, samples_s.value, width, height)
            img_w.value = to_png_bytes(arr)
            status.value = (f"az={az_s.value:.0f} el={el_s.value:.0f} dist={dist_s.value:.1f} "
                             f"(scene radius~{radius:.1f}) samples={samples_s.value}")
        except Exception as e:
            status.value = f"render failed: {e}"

    def _save(*_):
        status.value = 'saving snapshot for every method...'
        try:
            snap_dir = os.path.join(output_dir, 'novel_view_snapshots')
            os.makedirs(snap_dir, exist_ok=True)
            tag = f"az{az_s.value:.0f}_el{el_s.value:.0f}_dist{dist_s.value:.1f}"
            for name, model in methods.items():
                arr = render_novel_view(model, centroid, az_s.value, el_s.value, dist_s.value,
                                         samples_s.value, width, height)
                Image.fromarray((arr * 255).astype(np.uint8), mode='RGB').save(
                    os.path.join(snap_dir, f"{SCENE_NAME}_{tag}_{name}.png"))
            status.value = f"saved snapshots for {list(methods.keys())} to {snap_dir}"
        except Exception as e:
            status.value = f"save failed: {e}"

    for w in (az_s, el_s, dist_s, samples_s, method_dd):
        w.observe(_update, names='value')
    save_btn.on_click(_save)

    display(widgets.HBox([img_w, widgets.VBox([method_dd, az_s, el_s, dist_s, samples_s, save_btn, status])]))
    _update()


launch_scene_viewer(methods, positions.cpu())


What to look for here: pick an angle far from any training camera (the
`cameras.json` positions define roughly where the training cameras were --
if you plot them, angles well outside that ring are the genuinely novel
ones) and check each method at the same angle. Floating color blobs,
smeared/blurry regions, or structure that only resolves near the training
ring are the failure modes to note; none of that is captured by Part 1's
numbers, which only ever look at views close to a training camera.


## Later: a real interactive local viewer

Everything above is slider-based because Colab has no attached display.
When you move to a local machine with a GPU and a real window, use
`interactive_realtime_viewer.py` (next to this notebook's `rdv` package)
instead -- it renders every frame in response to actual mouse-look and
WASD movement, using OpenCV for the window/input loop (not Colab, not this
notebook). See that file's own header for usage; it needs the same
`point_cloud.ply` / `cameras.json` this notebook uses, but a local Vulkan
device instead of the Xvfb setup at the top of this notebook.
